# Credit Card Fraud Detection – ML Pipeline

This notebook explores the Kaggle transaction data and runs the repository's reproducible training workflow. Run Jupyter from the project root. The dataset is not included; see `data/README.md`.

## 1. Problem Statement

Classify transactions as legitimate (`Class = 0`) or fraud (`Class = 1`). Fraud is rare, so accuracy alone can hide poor fraud detection. We focus on precision, recall, F1-score, ROC-AUC, average precision, and the confusion matrix.

## 2. Import Libraries

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from src.preprocessing import split_dataset, validate_dataset
from src.train import run_training
from src.utils import load_dataset

## 3. Load Dataset

By default, the loader reads `data/creditcard.csv` and checks the target and numeric feature values.

In [ ]:
data = load_dataset(Path('data/creditcard.csv'))
display(data.head())
print(f'Rows: {len(data):,} | Columns: {data.shape[1]:,}')

## 4. Exploratory Data Analysis

In [ ]:
display(data.describe().T)
print('Missing values:', int(data.isna().sum().sum()))
print('Exact duplicate rows:', int(data.duplicated().sum()))

## 5. Target / Class Distribution

The class counts make the imbalance visible. The plot is descriptive and does not resample data.

In [ ]:
class_counts = data['Class'].value_counts().sort_index()
display(class_counts.rename(index={0: 'Legitimate (0)', 1: 'Fraud (1)'}).to_frame('transactions'))
axis = sns.countplot(data=data, x='Class')
axis.set(title='Transaction class distribution', xlabel='Class (0 = legitimate, 1 = fraud)', ylabel='Transactions')
plt.tight_layout()
plt.show()

## 6. Data Cleaning / Noise Handling

The project validates numeric, finite features and a binary target. It does not remove statistical outliers: unusual transactions may be informative, and no existing evidence supports deleting them. Duplicate counts are reported for inspection but rows are not silently removed.

In [ ]:
data = validate_dataset(data)
print('Validated rows retained:', len(data))

## 7. Feature Preparation

`Class` is the label; all remaining numeric columns are model features. No information from validation or test data is used to fit transformations.

In [ ]:
X = data.drop(columns='Class')
y = data['Class'].astype('int8')
print('Feature count:', X.shape[1])
print('Target prevalence:', y.mean())

## 8. Train / Validation / Test Split

A stratified 60/20/20 split preserves the rare-class proportion. The test partition remains untouched through model selection and threshold choice.

In [ ]:
splits = split_dataset(data, random_state=42)
for name, labels in [('train', splits.y_train), ('validation', splits.y_validation), ('test', splits.y_test)]:
    print(f'{name:10s}: {len(labels):,} rows, {int(labels.sum()):,} fraud ({labels.mean():.4%})')

## 9. Feature Scaling

`Time` and `Amount` are standardized inside the model pipeline. Their scaler is fitted during each training fit, including independently within each cross-validation fold. The PCA-derived `V1`–`V28` features pass through unchanged.

## 10. Class Imbalance Analysis

The training counts are the relevant counts for model fitting. Validation and test retain their original class distributions.

In [ ]:
display(splits.y_train.value_counts().sort_index().rename(index={0: 'Legitimate (0)', 1: 'Fraud (1)'}).to_frame('training rows'))

## 11. SMOTE

SMOTE creates synthetic minority examples during model fitting. It is a step after preprocessing in the imbalanced-learn pipeline, so it only sees a training partition. During cross-validation, each fold gets its own training-only resampling; validation and test data are never oversampled.

## 12. Baseline Model

The comparison begins with logistic regression trained without SMOTE, followed by logistic regression with training-only SMOTE. Validation metrics use a common 0.5 threshold.

## 13. XGBoost / Ensemble Model

The final candidate is XGBoost, a gradient-boosted tree ensemble, with SMOTE included in the training pipeline. No deep-learning model is used.

## 14. Hyperparameter Tuning

A stratified cross-validated grid search selects XGBoost parameters using F1-score. The search and final training use only train data until the validation-selected threshold is fixed.

In [ ]:
results = run_training(
    data_path='data/creditcard.csv',
    output_dir='results/run',
    random_state=42,
    cv_folds=3,
)
print('Best CV F1:', results['cross_validation']['best_mean_f1'])
print('Best parameters:', results['cross_validation']['best_parameters'])
print('Validation-selected threshold:', results['threshold_selection']['threshold'])

## 15. Evaluation

The following metrics are generated on the held-out test partition after tuning and threshold selection are complete.

In [ ]:
test_metrics = results['test_metrics']
display(pd.Series({key: value for key, value in test_metrics.items() if key not in {'confusion_matrix', 'confusion_matrix_labels'}}, name='test result').to_frame())

## 16. Confusion Matrix

In [ ]:
display(Image(filename='results/run/confusion_matrix.png'))

## 17. ROC Curve

ROC-AUC summarizes ranking performance across thresholds. On its own, it can be optimistic for a rare positive class, so it is read alongside precision-recall metrics.

## 18. Precision-Recall Analysis

Precision-recall behavior is especially useful when fraud cases are rare. The combined diagnostic figure includes both ROC and precision-recall curves.

In [ ]:
display(Image(filename='results/run/roc_precision_recall.png'))

## 19. Final Results

The generated values below describe this run's held-out test setup only. They are written to `results/run/metrics.json`; the validation comparison is saved separately in `results/run/validation_comparison.csv`. No resume metrics are inserted unless reproduced by this run.

In [ ]:
display(pd.read_csv('results/run/validation_comparison.csv'))
print('Test metrics:', json.dumps(results['test_metrics'], indent=2))

## 20. Conclusion

Compare fraud-class precision and recall with the operational cost of missed fraud versus unnecessary investigation. Treat the validation-selected threshold as a modeling choice, not a universal business optimum. Results here are educational holdout metrics, not production performance.